# House Prices — 고정 선형모델에 feature 엔지니어링으로 RMSE 낮추기 (Colab)

**연습 기법** (IOAI 2024 *Lost in Hyperspace* 와 동일): **모델은 고정(LinearRegression)** 해두고,
**오직 feature 엔지니어링**만으로 회귀 오차(RMSE)를 낮춘다. — '좋은 feature 가 좋은 모델보다 중요하다'.

**시나리오 매핑**:
- Lost in Hyperspace: 고정된 LinearRegression 에 구조화 데이터(5×5×5×6 배열)에서 feature 를 설계해 가중 RMSE 최소화.
- 여기(House Prices): 고정된 LinearRegression 에 주택 표 데이터에서 feature 를 설계해 **RMSE(log SalePrice)** 최소화.

**평가 지표**: 캐글 리더보드와 동일하게 `log(SalePrice)` 의 RMSE (낮을수록 좋음). 코드는 `pandas` + `scikit-learn` 기본만.


## 0. 라이브러리 설치 & Kaggle 자격증명


In [ ]:
import sys, subprocess
for pkg in ["kaggle", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")


In [ ]:
import os

# Kaggle API 토큰 (내장)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"

print("Kaggle 자격증명 설정 완료 (내장 토큰)")


## 1. Kaggle 에서 House Prices 데이터 다운로드


In [ ]:
import os, glob, zipfile

WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

# 대회 데이터 다운로드 + 압축 해제
api.competition_download_files("house-prices-advanced-regression-techniques", path=WORK_DIR, quiet=False)
for zpath in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(WORK_DIR)
    os.remove(zpath)

print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])


## 2. 데이터 준비 — 고정 모델 & 평가 지표
타깃은 `log(SalePrice)`(대회 지표가 로그 RMSE). **모델은 LinearRegression 으로 고정**하고, 이후 feature 만 바꾼다.


In [ ]:
import os, numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

train = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
y = np.log1p(train["SalePrice"].values)        # 대회 지표 = log RMSE
def rmse(a, b): return mean_squared_error(a, b) ** 0.5
print("train:", train.shape, " test:", test.shape)


## 3. Step 1 — 베이스라인: 숫자 feature 만
결측은 중앙값으로 채우고, 숫자형 컬럼만 그대로 사용.


In [ ]:
num = train.select_dtypes(exclude=["object"]).drop(columns=["Id", "SalePrice"])
X1 = num.fillna(num.median())
Xtr, Xva, ytr, yva = train_test_split(X1, y, test_size=0.2, random_state=42)
m = LinearRegression().fit(Xtr, ytr)
print(f"Step 1 (숫자 feature만)      RMSE(log) = {rmse(m.predict(Xva), yva):.5f}")


## 4. Step 2 — feature 엔지니어링 (모델은 그대로!)
- **공학 feature**: 총면적 `TotalSF`, 주택 나이 `Age`/`RemodAge`, 총 욕실 `TotalBath`
- **치우친 숫자 로그변환**(skew>1), **범주형 원핫 인코딩**(`get_dummies`)


In [ ]:
def engineer(df):
    df = df.copy()
    df["TotalSF"]  = df["TotalBsmtSF"].fillna(0) + df["1stFlrSF"] + df["2ndFlrSF"]
    df["Age"]      = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["TotalBath"] = (df["FullBath"] + 0.5 * df["HalfBath"]
                       + df["BsmtFullBath"].fillna(0) + 0.5 * df["BsmtHalfBath"].fillna(0))
    return df

def build_features(train_df, test_df):
    ntr = len(train_df)
    full = pd.concat([engineer(train_df).drop(columns=["SalePrice"]),
                      engineer(test_df)], ignore_index=True).drop(columns=["Id"])
    numc = full.select_dtypes(exclude=["object"]).copy()
    skew = numc.apply(lambda c: c.skew()).abs()           # 치우친 숫자 로그변환
    for c in skew[skew > 1].index:
        numc[c] = np.log1p(numc[c] - numc[c].min())
    catc = pd.get_dummies(full.select_dtypes(include=["object"]), dummy_na=True)
    X = pd.concat([numc.fillna(numc.median()), catc], axis=1)
    return X.iloc[:ntr], X.iloc[ntr:]

X2_tr, X2_te = build_features(train, test)
Xtr, Xva, ytr, yva = train_test_split(X2_tr, y, test_size=0.2, random_state=42)
m = LinearRegression().fit(Xtr, ytr)
print(f"Step 2 (feature 엔지니어링)  RMSE(log) = {rmse(m.predict(Xva), yva):.5f}")
print("→ 같은 LinearRegression 인데 feature 만 좋아져서 RMSE 가 내려간다 (Lost in Hyperspace 교훈).")


## 5. 최종 학습 & 캐글 제출 파일 생성
전체 train 으로 다시 학습한 뒤 test 를 예측 → `submission.csv` (Id, SalePrice).


In [ ]:
final = LinearRegression().fit(X2_tr, y)
pred = np.expm1(final.predict(X2_te))           # 로그 → 원래 가격으로 복원
submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"Id": test["Id"], "SalePrice": pred}).to_csv(submission_path, index=False)
print("Saved:", submission_path)
pd.read_csv(submission_path).head()


In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)


## 🏁 제출하기
`submission.csv` 를 [House Prices](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) 에 제출하세요.

**기법 요약**: 모델(LinearRegression)을 **고정**하고 feature 만 설계해 RMSE 를 낮췄습니다 — IOAI *Lost in Hyperspace* 와 동일한 패턴.
더 낮추려면: 더 많은 공학 feature(면적·품질 상호작용), 이상치 제거, 표준화 후 PCA, 범주형 타깃 인코딩 등을 시도해 보세요.
